# Compare a camera frame with an image loaded from disk

This notebook compares:

- a frame captured directly through a camera API; and
- an image loaded from a file and passed through the same analysis pipeline.

The camera APIs return processed image frames, not sensor RAW data.


In [3]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from jpeg_compression_detection.paths import CAMERA_OUTPUT_DIR, DATA_DIR, MODELS_DIR, display_path

CAMERA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_EXTENSION = ".png"
CAMERA_INDEX = 1

# Hardware access and image capture are opt-in so that Run All is safe on any host.
ENABLE_PRIMARY_OPENCV_CAPTURE = False
ENABLE_SECONDARY_OPENCV_CAPTURE = False
ENABLE_PYGAME_CAPTURE = False


## 1. Capture a frame with OpenCV


In [ ]:
def capture_opencv_frame(camera_index: int, output_stem: str):
    """Capture one processed camera frame and save lossless PPM and PNG copies."""
    print(f"Attempting to open camera index {camera_index}")
    capture = cv2.VideoCapture(camera_index)
    if not capture.isOpened():
        capture.release()
        print(f"Camera {camera_index} is unavailable; capture skipped.")
        return None

    try:
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = capture.get(cv2.CAP_PROP_FPS)
        print(f"Camera {camera_index} opened: {width}x{height}, {fps:.2f} FPS")

        # Discard a few frames while automatic exposure and white balance settle.
        for _ in range(10):
            capture.read()

        success, frame = capture.read()
        if not success or frame is None:
            print("Unable to capture a frame")
            return None

        print(f"Captured processed frame: shape={frame.shape}, dtype={frame.dtype}")
        print(f"Pixel range: {frame.min()} to {frame.max()}")

        ppm_path = CAMERA_OUTPUT_DIR / f"{output_stem}.ppm"
        png_path = CAMERA_OUTPUT_DIR / f"{output_stem}.png"
        cv2.imwrite(str(ppm_path), frame)
        cv2.imwrite(str(png_path), frame, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        print(f"Saved lossless PPM: {display_path(ppm_path)}")
        print(f"Saved lossless PNG: {display_path(png_path)}")

        ppm_loaded = cv2.imread(str(ppm_path))
        png_loaded = cv2.imread(str(png_path))
        print(f"PPM matches the captured frame: {np.array_equal(frame, ppm_loaded)}")
        print(f"PNG matches the captured frame: {np.array_equal(frame, png_loaded)}")
        return frame
    finally:
        capture.release()
        print(f"Camera {camera_index} released")


direct_camera_image = None
if ENABLE_PRIMARY_OPENCV_CAPTURE:
    direct_camera_image = capture_opencv_frame(
        CAMERA_INDEX,
        f"direct_camera_shot_cam{CAMERA_INDEX}",
    )
else:
    print("Primary OpenCV capture skipped. Set ENABLE_PRIMARY_OPENCV_CAPTURE = True to enable it.")


## 2. Load the saved frame from disk


In [ ]:
camera_frame_path = CAMERA_OUTPUT_DIR / f"direct_camera_shot_cam{CAMERA_INDEX}{FILE_EXTENSION}"
loaded_image = cv2.imread(str(camera_frame_path)) if camera_frame_path.is_file() else None

if loaded_image is None:
    loaded_image_rgb = None
    print(f"Saved camera frame is unavailable; load step skipped: {display_path(camera_frame_path)}")
else:
    loaded_image_rgb = cv2.cvtColor(loaded_image, cv2.COLOR_BGR2RGB)
    loaded_copy_path = CAMERA_OUTPUT_DIR / f"loaded_camera_shot{FILE_EXTENSION}"
    cv2.imwrite(str(loaded_copy_path), loaded_image)
    print(f"Loaded image: shape={loaded_image.shape}, dtype={loaded_image.dtype}")


## 3. Compare compression indicators


In [ ]:
from jpeg_compression_detection import detect_jpeg_compression


def print_image_indicators(image_path):
    """Print compression indicators for an image if it exists."""
    image_path = Path(image_path)
    if not image_path.is_file():
        print(f"Analysis skipped; image is missing: {display_path(image_path)}")
        return None

    print(f"Analyzing: {display_path(image_path)}")
    print("-" * 80)
    result = detect_jpeg_compression(image_path)
    if not isinstance(result, dict):
        print(f"Unable to compute indicators: {result}")
        return None

    print("\nCalculated indicators:")
    for key, value in result.items():
        if key not in {"filename", "extension"}:
            print(f"{key:20s}: {value:.4f}" if isinstance(value, float) else f"{key:20s}: {value}")
    print("-" * 80, "\n")
    return result


from pathlib import Path

print_image_indicators(CAMERA_OUTPUT_DIR / "direct_camera_shot_cam1.png")
print_image_indicators(DATA_DIR / "test-images" / "1100x1100-woman.png")


## 4. Load the format-classification model


In [ ]:
import joblib

model_path = MODELS_DIR / "best_classification_model.pkl"
scaler_path = MODELS_DIR / "scaler.pkl"
feature_columns_path = MODELS_DIR / "feature_columns.pkl"

loaded_model = loaded_scaler = loaded_feature_columns = None
required_model_files = [model_path, scaler_path, feature_columns_path]
if all(path.is_file() for path in required_model_files):
    loaded_model = joblib.load(model_path)
    loaded_scaler = joblib.load(scaler_path)
    loaded_feature_columns = joblib.load(feature_columns_path)
    print("Model loaded successfully!")
    print(f"Model type: {type(loaded_model).__name__}")
    print(f"Number of features: {len(loaded_feature_columns)}")
else:
    missing = [display_path(path) for path in required_model_files if not path.is_file()]
    print("Model loading skipped; missing files: " + ", ".join(missing))


## 5. Classify images with the trained model


In [37]:
import pandas as pd


def classify_image(image_path):
    """Calculate indicators and classify an existing image as PPM or JFIF."""
    image_path = Path(image_path)
    if not image_path.is_file():
        print(f"Classification skipped; image is missing: {display_path(image_path)}")
        return None
    if any(item is None for item in (loaded_model, loaded_scaler, loaded_feature_columns)):
        print("Classification skipped because the model files are unavailable")
        return None

    print(f"Analyzing: {display_path(image_path)}")
    print("-" * 80)
    result = detect_jpeg_compression(image_path)
    if not isinstance(result, dict):
        print(f"Unable to compute indicators: {result}")
        return None

    print("\nCalculated indicators:")
    for key, value in result.items():
        if key not in {"filename", "extension", "estimated_quality"} and isinstance(value, float):
            print(f"{key:20s}: {value:.4f}")

    feature_row = pd.DataFrame([{column: result.get(column, 0) for column in loaded_feature_columns}])
    if feature_row.isnull().any().any():
        print("\nMissing indicator values:")
        print(feature_row.isnull().sum()[feature_row.isnull().sum() > 0])
        return None

    scaled_row = loaded_scaler.transform(feature_row)
    prediction = loaded_model.predict(scaled_row)[0]
    probabilities = loaded_model.predict_proba(scaled_row)[0]
    print("\n" + "-" * 80)
    print("Prediction")
    print("-" * 80)
    print(f"Format: {'PPM (uncompressed)' if prediction == 1 else 'JFIF (JPEG compression)'}")
    print(f"\nProbabilities:\n  JFIF: {probabilities[0] * 100:6.2f}%\n  PPM:  {probabilities[1] * 100:6.2f}%")
    print(f"\nConfidence: {max(probabilities) * 100:.1f}%")
    return {"prediction": prediction, "probabilities": probabilities}


classify_image(CAMERA_OUTPUT_DIR / "direct_camera_shot_cam1.png")
classify_image(DATA_DIR / "test-images" / "1100x1100-woman.png")
classify_image(CAMERA_OUTPUT_DIR / "camera_button_photo.jpg")


In [ ]:
# The classification calls are included in the helper-definition cell above.


## 6. Optional capture from a second OpenCV camera


In [ ]:
SECONDARY_CAMERA_INDEX = 0
secondary_camera_path = CAMERA_OUTPUT_DIR / f"direct_camera_shot_cam{SECONDARY_CAMERA_INDEX}.bmp"

if ENABLE_SECONDARY_OPENCV_CAPTURE:
    secondary_frame = capture_opencv_frame(
        SECONDARY_CAMERA_INDEX,
        f"direct_camera_shot_cam{SECONDARY_CAMERA_INDEX}",
    )
    if secondary_frame is not None:
        cv2.imwrite(str(secondary_camera_path), secondary_frame)
        print(f"Saved lossless BMP: {display_path(secondary_camera_path)}")
else:
    print("Secondary OpenCV capture skipped. Set ENABLE_SECONDARY_OPENCV_CAPTURE = True to enable it.")


In [ ]:
classify_image(secondary_camera_path)


## 7. Optional capture with pygame


In [63]:
pygame_photo_path = CAMERA_OUTPUT_DIR / "simple_photo.bmp"

if not ENABLE_PYGAME_CAPTURE:
    print("pygame capture skipped. Set ENABLE_PYGAME_CAPTURE = True to enable it.")
else:
    try:
        import pygame
        import pygame.camera

        pygame.camera.init()
        cameras = pygame.camera.list_cameras()
        print(f"Available cameras: {cameras}")
        if not cameras:
            print("No pygame-compatible camera is available")
        else:
            webcam = pygame.camera.Camera(cameras[0], (640, 480))
            webcam.start()
            try:
                surface = webcam.get_image()
                image_bytes = pygame.image.tostring(surface, "RGB")
                image = Image.frombytes("RGB", surface.get_size(), image_bytes)
                image.save(pygame_photo_path)
                print(f"Photo saved as BMP: {display_path(pygame_photo_path)}")
            finally:
                webcam.stop()
    except Exception as exc:
        print(f"pygame capture unavailable: {exc}")


pygame 2.5.2 (SDL 2.28.3, Python 3.10.11)
Hello from the pygame community. https://www.pygame.org/contribute.html
Available cameras: [0, 1]
Available cameras: [0, 1]
Photo saved as BMP
Photo saved as BMP


In [ ]:
classify_image(pygame_photo_path)
